# Valuation-to-Option Pricing and Greeks Lab: Lockheed Martin (LMT)

**Contract analysed:** LMT June 17, 2027 $580 American put (`LMT270617P00580000`), contract multiplier 100.
**Valuation date:** November 9, 2026 (spot and option quotes captured at the same time, about 12:35 PM ET).

| Version | What it adds | Where |
|---|---|---|
| v1: initial model | Course template: continuous dividend yield, 200-step CRR, five analyst states | Sections 1–8 (template code, unchanged except blank-safe DCF/comps inputs) |
| v2: model improvement | Discrete, dated LMT dividends (escrowed-dividend CRR), American-exercise checks, implied volatility, horizon-alignment test | Section 9 |

# Valuation-to-Option Pricing Lab

This notebook keeps two valuation tracks separate:

1. **Analyst-scenario discounted expected payoff:** five terminal equity-value scenarios with subjective weights.
2. **CRR risk-neutral lattice value:** a no-arbitrage option-pricing model built through repeated two-state replication.

The five analyst scenarios are not an expanded binomial tree and are not risk-neutral probabilities. They answer a different question: *Given these valuation beliefs and weights, what is the discounted expected option payoff?*

## 1. Load and validate student inputs

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask']: M[k]=float(M[k])
# Student valuation inputs may be left blank until entered; blanks are exported as null, never as 0.
for k in ['dcf_value','comps_value']: M[k]=None if pd.isna(M[k]) or str(M[k]).strip()=='' else float(M[k])
if M['dcf_value'] is None: print('WARNING: dcf_value is blank. Enter the DCF equity value per diluted share in data/company_inputs.csv.')
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')
M

{'company': 'Lockheed Martin Corporation',
 'ticker': 'LMT',
 'valuation_date': '2026-11-09',
 'spot': 586.63,
 'strike': 580.0,
 'maturity_years': 0.6027,
 'risk_free_rate': 0.0416,
 'dividend_yield': 0.0245,
 'volatility': 0.3036,
 'volatility_type': 'implied',
 'option_type': 'put',
 'exercise_style': 'american',
 'lattice_steps': 200,
 'market_bid': 48.7,
 'market_ask': 51.6,
 'dcf_value': None,
 'comps_value': None,
 'market_mid': 50.150000000000006}

## 2. Equity valuation and analyst distribution

In [2]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
valuation={k:v for k,v in valuation.items() if v is not None}  # omit blank student inputs instead of plotting them as $0
pd.Series(valuation)

Current market    586.63
Analyst median    650.00
dtype: float64

## 3. Five analyst-informed valuation states

The state prices are reproducible quantiles of the analyst-target distribution. The weights are disclosed subjective scenario masses, not risk-neutral probabilities.

- Severe bear: 5th percentile, 10% weight
- Bear: 25th percentile, 20% weight
- Base: 50th percentile, 40% weight
- Bull: 75th percentile, 20% weight
- Tail: 95th percentile, 10% weight

In [3]:
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}
weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({
    'state':list(quantile_map),
    'quantile':list(quantile_map.values()),
    'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],
    'subjective_weight':[weights[k] for k in quantile_map]
})
assert np.isclose(states.subjective_weight.sum(),1.0)
states

,state,quantile,terminal_price,subjective_weight
0,Severe bear,0.05,562.8,0.1
1,Bear,0.25,600.0,0.2
2,Base,0.50,650.0,0.4
3,Bull,0.75,696.0,0.2
4,Tail,0.95,714.7,0.1


## 4. Analyst-scenario payoff decomposition

In [4]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum())
discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years']))
belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum())
weight_zero=1-weight_positive
print(f'Weight producing zero payoff: {weight_zero:.1%}')
print(f'Weight producing positive payoff: {weight_positive:.1%}')
print(f'Weighted terminal payoff: ${weighted_terminal_payoff:,.2f}')
print(f'Discount factor: {discount_factor:.4f}')
print(f'Analyst-scenario discounted expected payoff: ${belief_value:,.2f}')
states

Weight producing zero payoff: 90.0%
Weight producing positive payoff: 10.0%
Weighted terminal payoff: $1.72
Discount factor: 0.9752
Analyst-scenario discounted expected payoff: $1.68


,state,quantile,terminal_price,subjective_weight,option_payoff,weighted_payoff
0,Severe bear,0.05,562.8,0.1,17.2,1.72
1,Bear,0.25,600.0,0.2,0.0,0.00
2,Base,0.50,650.0,0.4,0.0,0.00
3,Bull,0.75,696.0,0.2,0.0,0.00
4,Tail,0.95,714.7,0.1,0.0,0.00


### Interpretation
A low analyst-scenario value is intuitive when much of the assigned weight ends below the call strike or above the put strike. The payoff contribution is visible state by state. This is not a replication price.

## 5. CRR risk-neutral lattice

In [5]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

{'price': 48.32861462990032,
 'u': 1.0168058861535765,
 'd': 0.98347188349081,
 'p_star': 0.49737947328618226,
 'dt': 0.0030135}

## 6. Theoretical validation and distribution comparison

In [6]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]
conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_log_dispersion=float(np.log(targets.target/M['spot']).std(ddof=1))
vol_horizon_dispersion=float(M['volatility']*np.sqrt(M['maturity_years']))
distribution_message='Analyst targets are narrower than the volatility input used by CRR.' if analyst_log_dispersion<vol_horizon_dispersion else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
print(distribution_message)
validation

Analyst targets are narrower than the volatility input used by CRR.


{'valid_risk_neutral_probability': True,
 'inside_theoretical_bounds': True,
 'put_call_parity_pass': True,
 'convergence_pass': True,
 'parity_residual': -4.995115432393504e-12,
 'convergence_gap': -0.026367923351337197}

## 7. Export dashboard data

In [7]:
results={
 'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,
 'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,
 'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},
 'distribution_comparison':{'analyst_log_dispersion':analyst_log_dispersion,'crr_horizon_volatility':vol_horizon_dispersion,'message':distribution_message},
 'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],
 'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}
}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Exported docs/model_results.json')
results['comparison']

Exported docs/model_results.json


{'Analyst-scenario discounted expected payoff': 1.6774117338854517,
 'CRR risk-neutral lattice value': 48.32861462990032,
 'Observed market midpoint': 50.150000000000006}

## 8. Decision memo prompt

Explain why the analyst-scenario payoff and CRR value differ. Address the share of scenario weight producing a zero payoff, the contribution from positive-payoff states, the strike, analyst-target dispersion, the CRR volatility input, and why neither estimate should automatically be characterized as the uniquely correct market price.

## 9. Model improvement (v2): discrete, dated dividends

**What changed.** The initial model (Sections 1–8, kept exactly as run) treats LMT's dividend as a continuous yield of 2.45%. LMT actually pays a fixed **$3.60 per quarter**. Three ex-dividend dates fall before the June 17, 2027 expiration: **Dec 1, 2026** (declared), plus **Mar 1, 2027** and **Jun 1, 2027** (estimated from LMT's usual quarterly cycle; see `data/dividends.csv`).

**Why it matters for this contract.** A put gains value when the stock drops on an ex-dividend date. For an *American* put, the timing also changes the early-exercise decision: just before a cash dividend, the holder is better off waiting. A smooth yield spreads the same cash evenly over time and misses both effects.

**Method (escrowed-dividend CRR).** The present value of the dividends paid before expiration is removed from the spot price. The lattice is built on the remaining "risky" component, S* = S − PV(dividends). At every node, the stock price used for the early-exercise test adds back the PV of the dividends still to come. All other inputs (spot, strike, rate, volatility, valuation date) are unchanged, so the only difference between v1 and v2 is how dividends are treated.

**What the improvement affects:** the dividend assumption, the CRR value, the American early-exercise premium, the implied volatility backed out of the market midpoint, the new validation checks below, and every Greek in Sections 10–14 (all of them use the v2 model).

In [8]:
import math
DIV=pd.read_csv(DATA/'dividends.csv')
for col in ['ex_date','amount','status']:
    if col not in DIV.columns: raise ValueError(f'dividends.csv is missing column {col}.')
DIV['ex_date']=pd.to_datetime(DIV['ex_date'],format='%Y-%m-%d')
if not DIV.amount.gt(0).all(): raise ValueError('Dividend amounts must be positive dollars per share.')
VAL_DATE=pd.Timestamp(M['valuation_date'])
EXPIRY=pd.Timestamp('2027-06-17')
CONTRACT_MULTIPLIER=100
T_days=(EXPIRY-VAL_DATE).days
assert abs(T_days/365-M['maturity_years'])<0.001, 'maturity_years must equal calendar days to expiration / 365'

def dividend_schedule(shift_years=0.0, T=None):
    """(time in years from the valuation date, amount) for ex-dates strictly between now and expiration.
    shift_years moves the valuation date forward (used for theta and time-decay repricing)."""
    T=M['maturity_years'] if T is None else T
    t=(DIV.ex_date-VAL_DATE).dt.days/365-shift_years
    keep=(t>0)&(t<T)
    return [(float(a),float(b)) for a,b in zip(t[keep],DIV.amount[keep])]

def pv_dividends(divs,r,t=0.0):
    return sum(D*math.exp(-r*(td-t)) for td,D in divs if td>t+1e-12)

def crr_discrete(spot,strike,T,r,sigma,N,divs,kind='put',american=True,keep_nodes=False):
    """Escrowed-dividend CRR lattice (model v2). Returns price, lattice parameters and, optionally, the first three time steps."""
    divs=[(td,D) for td,D in divs if 0<td<T]
    pv0=pv_dividends(divs,r,0.0); S_star=spot-pv0
    dt=T/N; u=math.exp(sigma*math.sqrt(dt)); d=1/u; growth=math.exp(r*dt); p=(growth-d)/(u-d); disc=1/growth
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); values=payoff(S_star*u**j*d**(N-j),strike,kind)
    nodes={}; early_nodes=0
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        jj=np.arange(n+1); stock=S_star*u**jj*d**(n-jj)+pv_dividends(divs,r,n*dt)
        if american:
            exercise=payoff(stock,strike,kind); early_nodes+=int((exercise>values+1e-12).sum()); values=np.maximum(values,exercise)
        if keep_nodes and n<=2: nodes[n]=(stock.copy(),values.copy())
    return {'price':float(values[0]),'u':u,'d':d,'p_star':p,'dt':dt,'S_star':S_star,'pv_dividends':pv0,'early_exercise_nodes':early_nodes,'nodes':nodes}

V2_STEPS=500
def v2_price(spot=None,sigma=None,r=None,shift_years=0.0,N=V2_STEPS,american=True,kind=None):
    """Model-v2 value: average of the N- and (N+1)-step lattices, which removes most of the odd/even oscillation of CRR."""
    spot=M['spot'] if spot is None else spot; sigma=M['volatility'] if sigma is None else sigma
    r=M['risk_free_rate'] if r is None else r; kind=M['option_type'] if kind is None else kind
    T=M['maturity_years']-shift_years; divs=dividend_schedule(shift_years,T)
    return 0.5*(crr_discrete(spot,M['strike'],T,r,sigma,N,divs,kind,american)['price']+crr_discrete(spot,M['strike'],T,r,sigma,N+1,divs,kind,american)['price'])

divs0=dividend_schedule()
v2=crr_discrete(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],V2_STEPS,divs0,M['option_type'],M['exercise_style']=='american')
v2_value=v2_price()
v2_european=v2_price(american=False)
v1_european=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],V2_STEPS,M['dividend_yield'],M['option_type'],False)['price']
v1_american_500=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],V2_STEPS,M['dividend_yield'],M['option_type'],True)['price']
print('Dividends inside the option life:',[(round(t,4),D) for t,D in divs0])
print(f"PV of dividends: ${v2['pv_dividends']:.2f}   escrowed spot S* = ${v2['S_star']:.2f}")
progression=pd.DataFrame({
 'model':['v1 initial (template, 200 steps)','v1 initial (500 steps)','v2 improved (500/501-step average)'],
 'dividend_treatment':['continuous yield 2.45%','continuous yield 2.45%','3 discrete $3.60 cash dividends'],
 'american_value':[lattice['price'],v1_american_500,v2_value],
 'european_value':[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],False)['price'],v1_european,v2_european]})
progression['early_exercise_premium']=progression.american_value-progression.european_value
progression['gap_to_market_mid']=M['market_mid']-progression.american_value
progression.round(4)

Dividends inside the option life: [(0.0603, 3.6), (0.3068, 3.6), (0.5589, 3.6)]
PV of dividends: $10.66   escrowed spot S* = $575.97


,model,dividend_treatment,american_value,european_value,early_exercise_premium,gap_to_market_mid
0,"v1 initial (template, 200 steps)",continuous yield 2.45%,48.3286,47.7267,0.6019,1.8214
1,v1 initial (500 steps),continuous yield 2.45%,48.3008,47.7004,0.6004,1.8492
2,v2 improved (500/501-step average),3 discrete $3.60 cash dividends,49.0512,48.5522,0.4989,1.0988


### 9.1 American-exercise convergence and implied volatility

The template's convergence check uses the European lattice, so this repeats the test for the American v2 model at 25, 50, 100, 200 and 500 steps (plus 1,000 as a reference). CRR prices oscillate between odd and even step counts, which is why the reported v2 value averages N and N+1.

The vendor's implied volatility (30.36%) comes from an undisclosed model. As a check, I back out the volatility that makes the **v2 American lattice** equal the market bid, midpoint and ask.

In [9]:
conv_v2=[]
for n in [25,50,100,200,500,1000]:
    raw=crr_discrete(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,divs0,M['option_type'],True)['price']
    conv_v2.append({'steps':n,'raw_price':raw,'avg_n_n1':v2_price(N=n)})
conv_v2=pd.DataFrame(conv_v2)
conv_v2['change_vs_previous']=conv_v2.avg_n_n1.diff()

def implied_vol_v2(target,lo=0.05,hi=1.5,N=300):
    f=lambda s: v2_price(sigma=s,N=N)-target
    if f(lo)>0 or f(hi)<0: return float('nan')
    for _ in range(60):
        mid=0.5*(lo+hi)
        if f(mid)>0: hi=mid
        else: lo=mid
    return 0.5*(lo+hi)
iv={'bid':implied_vol_v2(M['market_bid']),'mid':implied_vol_v2(M['market_mid']),'ask':implied_vol_v2(M['market_ask'])}
print('v2 implied volatility  bid {bid:.2%}  mid {mid:.2%}  ask {ask:.2%}   vendor IV {v:.2%}'.format(**iv,v=M['volatility']))
conv_v2.round(4)

v2 implied volatility  bid 30.16%  mid 30.98%  ask 31.81%   vendor IV 30.36%


,steps,raw_price,avg_n_n1,change_vs_previous
0,25,49.4523,49.1106,NaN
1,50,48.9928,49.1193,0.0086
2,100,49.0646,49.1010,-0.0182
3,200,49.0665,49.0698,-0.0312
4,500,49.0608,49.0512,-0.0186
5,1000,49.0511,49.0407,-0.0105


### 9.2 Horizon alignment and an alternative set of states (disclosed, not used as the primary result)

The analyst targets are 12-month opinions, but the option expires in 0.60 years, a gap of about 0.40 years (more than the 0.25-year tolerance). The primary five states (Section 3) keep the course's required quantile construction. As a disclosed alternative, each target is scaled to the option's horizon by assuming the implied move is earned at a constant rate: S_T = S0 × (target / S0)^(T / 1 year). The weights stay the same.

In [10]:
horizon_gap=float(targets.horizon_months.iloc[0]/12-M['maturity_years'])
alt=states[['state','quantile','subjective_weight']].copy()
alt['terminal_price_12m']=states.terminal_price
alt['terminal_price_horizon_scaled']=M['spot']*(states.terminal_price/M['spot'])**(M['maturity_years']/(targets.horizon_months.iloc[0]/12))
alt['payoff_scaled']=payoff(alt.terminal_price_horizon_scaled,M['strike'],M['option_type'])
alt['weighted_payoff_scaled']=alt.subjective_weight*alt.payoff_scaled
alt_belief_value=float(discount_factor*alt.weighted_payoff_scaled.sum())
print(f'Horizon gap: {horizon_gap:.2f} years -> alignment warning: {abs(horizon_gap)>0.25}')
print(f'Primary five-state discounted payoff: ${belief_value:.2f}   horizon-scaled alternative: ${alt_belief_value:.2f}')
alt.round(2)

Horizon gap: 0.40 years -> alignment warning: True
Primary five-state discounted payoff: $1.68   horizon-scaled alternative: $0.77


,state,quantile,subjective_weight,terminal_price_12m,terminal_price_horizon_scaled,payoff_scaled,weighted_payoff_scaled
0,Severe bear,0.05,0.1,562.8,572.15,7.85,0.79
1,Bear,0.25,0.2,600.0,594.65,0.00,0.00
2,Base,0.50,0.4,650.0,624.04,0.00,0.00
3,Bull,0.75,0.2,696.0,650.30,0.00,0.00
4,Tail,0.95,0.1,714.7,660.77,0.00,0.00


### 9.3 Additional model checks (v2)

In [11]:
v2_checks={
 'valid_risk_neutral_probability_v2':bool(0<v2['p_star']<1),
 'american_at_least_european':bool(v2_value>=v2_european-1e-9),
 'american_put_at_least_intrinsic':bool(v2_value>=max(M['strike']-M['spot'],0)),
 'put_below_strike':bool(v2_value<=M['strike']),
 'bid_not_above_ask':bool(M['market_bid']<=M['market_ask']),
 'rate_and_vol_in_decimal_form':bool(0<M['risk_free_rate']<0.2 and 0.01<M['volatility']<2),
 'american_convergence_pass':bool(abs(conv_v2.avg_n_n1.iloc[-2]-conv_v2.avg_n_n1.iloc[-3])<0.05),
 'escrowed_spot_positive':bool(v2['S_star']>0),
}
warnings_v2={
 'analyst_horizon_alignment':'WARNING: 12-month targets vs 0.60-year option (gap {:.2f} years > 0.25). Disclosed; see 9.2.'.format(horizon_gap) if abs(horizon_gap)>0.25 else 'OK',
 'wide_spread':'WARNING: bid-ask spread is {:.1%} of the midpoint; the midpoint is a mechanical reference, not an executable price.'.format((M['market_ask']-M['market_bid'])/M['market_mid']) if (M['market_ask']-M['market_bid'])/M['market_mid']>0.03 else 'OK',
 'estimated_dividends':'NOTE: {} of {} dividends inside the option life are estimated, not declared.'.format(int((DIV.status=='estimated').sum()),len(DIV)),
}
assert all(v2_checks.values()), v2_checks
for k,v in warnings_v2.items(): print(k,'->',v)
v2_checks

analyst_horizon_alignment -> WARNING: 12-month targets vs 0.60-year option (gap 0.40 years > 0.25). Disclosed; see 9.2.
wide_spread -> WARNING: bid-ask spread is 5.8% of the midpoint; the midpoint is a mechanical reference, not an executable price.
estimated_dividends -> NOTE: 2 of 3 dividends inside the option life are estimated, not declared.


{'valid_risk_neutral_probability_v2': True,
 'american_at_least_european': True,
 'american_put_at_least_intrinsic': True,
 'put_below_strike': True,
 'bid_not_above_ask': True,
 'rate_and_vol_in_decimal_form': True,
 'american_convergence_pass': True,
 'escrowed_spot_positive': True}

In [12]:
improvement={
 'description':'Replaced the continuous 2.45% dividend yield with three discrete, dated $3.60 cash dividends (escrowed-dividend CRR); added American convergence, early-exercise, implied-volatility and horizon checks.',
 'dividends':[{'time_years':t,'amount':D} for t,D in divs0],'pv_dividends':v2['pv_dividends'],'escrowed_spot':v2['S_star'],
 'steps':V2_STEPS,'v2_value':v2_value,'v2_european':v2_european,'early_exercise_premium_v2':v2_value-v2_european,
 'v1_value':lattice['price'],'v1_european':float(progression.european_value.iloc[0]),
 'progression':progression.to_dict('records'),'convergence_american_v2':conv_v2.fillna(0).to_dict('records'),
 'implied_vol_v2':iv,'vendor_iv':M['volatility'],'horizon_gap_years':horizon_gap,
 'alternative_states':alt.to_dict('records'),'alternative_belief_value':alt_belief_value,
 'checks':v2_checks,'warnings':warnings_v2,'u':v2['u'],'d':v2['d'],'p_star':v2['p_star'],'dt':v2['dt']}
results['improvement']=improvement
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Updated docs/model_results.json with the v2 improvement')

Updated docs/model_results.json with the v2 improvement
